**Figure 2: Tail Latency and Real-Agent Behavior.** (a) per-call latency percentiles
on the deterministic workload (log scale); (b) end-to-end wall latency of the real
DeepSeek refactor (3 repeats, success 100%, host leaks 0). Results suggest that read
tracing is a tail phenomenon (x2.2 on p95 versus x1.3 on p50) and that model latency
dominates the deployed experience, making the deterministic workload the honest
worst case.


In [ ]:
# ipython -c "%run plot_tail.ipynb"
import json
# FAST/USENIX line-plot conventions: white panels, boxed top legend, red solid squares = ours.
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, cm

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

OURS  = dict(color='#c00000', marker='s', linestyle='-',  linewidth=1.0, markersize=3.2)
BASE1 = dict(color='#e78129', marker='x', linestyle=':',  linewidth=0.9, markersize=3.6, markeredgewidth=0.9)
BASE2 = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
REF   = dict(color='black', linestyle='--', linewidth=0.8)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

with open(RESULTS / 'robustness.json', 'r', encoding='utf-8') as handle:
    rows = json.load(handle)
with open(RESULTS / 'real_agent_robustness.json', 'r', encoding='utf-8') as handle:
    real_agent = json.load(handle)
tail = pd.DataFrame([r for r in rows if r.get('suite') == 'p50_p95']).set_index('mode')

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(3.4)))

# (a) Percentile curves per mode, log y.
ax0 = plt.subplot(1, 2, 1)
xs = [0, 1]
nt = [float(tail.loc['agenttx_without_read_tracing', c]) for c in ['step_p50_ms', 'step_p95_ms']]
fl = [float(tail.loc['agenttx_full', c]) for c in ['step_p50_ms', 'step_p95_ms']]
line_nt, = ax0.plot(xs, nt, **BASE2, label='AgentTX no-trace')
line_fl, = ax0.plot(xs, fl, **OURS, label='AgentTX full (ours)')
ax0.annotate(f"x{fl[1] / nt[1]:.1f}", (1, fl[1]), textcoords='offset points', xytext=(-4, 5), ha='right', fontsize=6, color=OURS['color'])
ax0.set_yscale('log')
ax0.set_yticks([20, 50, 100, 200, 500, 1000])
ax0.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax0.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
ax0.set_xticks(xs, labels=['p50', 'p95'], fontsize=7)
ax0.set_xlim(-0.35, 1.35)
ax0.set_ylabel('Per-call latency (ms)', fontsize=8)
ax0.set_xlabel('(a) Deterministic workload tail', fontsize=8)
ax0.tick_params(axis='y', labelsize=7)

# (b) Real-agent wall latency percentiles.
ax1 = plt.subplot(1, 2, 2)
wall = [float(real_agent['wall_p50_s']), float(real_agent['wall_p95_s'])]
ax1.plot(xs, wall, **OURS)
ax1.set_xticks(xs, labels=['p50', 'p95'], fontsize=7)
ax1.set_xlim(-0.35, 1.35)
ax1.set_ylim(0, max(wall) * 1.25)
ax1.set_ylabel('Task latency (s)', fontsize=8)
ax1.set_xlabel('(b) Real-agent refactor', fontsize=8)
ax1.tick_params(axis='y', labelsize=7)
ax1.text(0.5, 0.88, f"success={real_agent['success_rate']:.0%}, leak={real_agent['host_leak_rate']:.0%}",
         transform=ax1.transAxes, ha='center', fontsize=7)

fig.legend(handles=[line_nt, line_fl], loc='upper center', bbox_to_anchor=(0.5, 1.06), ncol=2,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.4, borderpad=0.3)
plt.tight_layout(pad=0.4, rect=[0.0, 0.0, 1.0, 0.93])
plt.savefig(FIGDIR / 'FIG-Motivation-Tail.pdf', bbox_inches='tight', pad_inches=0.02)
plt.savefig(FIGDIR / 'FIG-Motivation-Tail.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

print(f"read tracing tail: p50 x{fl[0] / nt[0]:.2f}, p95 x{fl[1] / nt[1]:.2f}")
